In [1]:
!pip install -q -U "transformers" "datasets" "tokenizers" "accelerate"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.6 MB/s eta 0:00:00


In [2]:
import os
import json
import math
import random

import numpy as np
import torch

from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LambdaLR

from datasets import load_from_disk

from transformers import (
    BertConfig,
    BertForMaskedLM,
    PreTrainedTokenizerFast,
    DataCollatorForLanguageModeling
)

In [3]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA capability:",
        torch.cuda.get_device_capability(0)
    )

    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA capability: (7, 5)
GPU memory: 14.56 GB


In [4]:
BASE_PATH = (
    "/kaggle/input/datasets/"
    "belovedorange/"
    "indian-legal-slm-tokenized-2048"
)

DATASET_PATH = os.path.join(
    BASE_PATH,
    "indian_legal_2048"
)

TOKENIZER_PATH = os.path.join(
    BASE_PATH,
    "indian_legal_tokenizer",
    "tokenizer.json"
)

print("Dataset path:")
print(DATASET_PATH)

print("\nTokenizer path:")
print(TOKENIZER_PATH)

Dataset path:
/kaggle/input/datasets/belovedorange/indian-legal-slm-tokenized-2048/indian_legal_2048

Tokenizer path:
/kaggle/input/datasets/belovedorange/indian-legal-slm-tokenized-2048/indian_legal_tokenizer/tokenizer.json


In [5]:
dataset = load_from_disk(
    DATASET_PATH
)

print(dataset)

print(
    "\nTraining examples:",
    len(dataset["train"])
)

print(
    "Validation examples:",
    len(dataset["validation"])
)

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 155060
    })
    validation: Dataset({
        features: ['input_ids'],
        num_rows: 17474
    })
})

Training examples: 155060
Validation examples: 17474


In [6]:
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=TOKENIZER_PATH,

    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<bos>",
    eos_token="<eos>",
    mask_token="[MASK]"
)

print(
    "Vocabulary:",
    tokenizer.vocab_size
)

print(
    "PAD token ID:",
    tokenizer.pad_token_id
)

print(
    "MASK token ID:",
    tokenizer.mask_token_id
)

VOCAB_SIZE = len(tokenizer)

print(
    "Final vocabulary size:",
    VOCAB_SIZE
)

Vocabulary: 16000
PAD token ID: 0
MASK token ID: 16000
Final vocabulary size: 16001


In [7]:
text = "The Supreme Court of India"

encoded = tokenizer(
    text,
    return_tensors="pt"
)

print(
    "Token IDs:"
)

print(
    encoded["input_ids"]
)

print(
    "\nDecoded:"
)

print(
    tokenizer.decode(
        encoded["input_ids"][0]
    )
)

Token IDs:
tensor([[377, 909, 191,  83, 646]])

Decoded:
ĠThe ĠSupreme ĠCourt Ġof ĠIndia


In [8]:
D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 6
D_FF = 2048
CONTEXT_LENGTH = 2048

config = BertConfig(

    vocab_size=VOCAB_SIZE,

    hidden_size=D_MODEL,

    num_hidden_layers=NUM_LAYERS,

    num_attention_heads=NUM_HEADS,

    intermediate_size=D_FF,

    max_position_embeddings=CONTEXT_LENGTH,

    hidden_dropout_prob=0.1,

    attention_probs_dropout_prob=0.1,

    pad_token_id=tokenizer.pad_token_id
)

print(config)

BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 512,
  "initializer_range": 0.02,
  "intermediate_size": 2048,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 2048,
  "model_type": "bert",
  "num_attention_heads": 8,
  "num_hidden_layers": 6,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.15.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 16001
}



In [9]:
model = BertForMaskedLM(
    config
)

print(
    "Parameters:",
    f"{model.num_parameters():,}"
)

Parameters: 28,437,121


In [10]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(
    DEVICE
)

print(
    "Model device:",
    next(model.parameters()).device
)

Model device: cuda:0


In [11]:
data_collator = DataCollatorForLanguageModeling(

    tokenizer=tokenizer,

    mlm=True,

    mlm_probability=0.15
)

In [12]:
BATCH_SIZE = 1

train_loader = DataLoader(

    dataset["train"],

    batch_size=BATCH_SIZE,

    shuffle=True,

    collate_fn=data_collator,

    num_workers=2,

    pin_memory=True
)

validation_loader = DataLoader(

    dataset["validation"],

    batch_size=BATCH_SIZE,

    shuffle=False,

    collate_fn=data_collator,

    num_workers=2,

    pin_memory=True
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

Training batches: 155060
Validation batches: 17474


In [13]:
test_batch = next(
    iter(train_loader)
)

print(
    "Input IDs:",
    test_batch["input_ids"].shape
)

print(
    "Attention mask:",
    test_batch["attention_mask"].shape
)

print(
    "Labels:",
    test_batch["labels"].shape
)

Input IDs: torch.Size([1, 2048])
Attention mask: torch.Size([1, 2048])
Labels: torch.Size([1, 2048])


In [14]:
labels_cpu = test_batch["labels"]

masked_tokens = (
    labels_cpu != -100
).sum().item()

total_tokens = (
    labels_cpu.numel()
)

print(
    "Total tokens:",
    total_tokens
)

print(
    "Masked target tokens:",
    masked_tokens
)

print(
    "Mask percentage:",
    100 * masked_tokens / total_tokens
)

Total tokens: 2048
Masked target tokens: 311
Mask percentage: 15.185546875


In [15]:
model.eval()

input_ids = (
    test_batch["input_ids"]
    .to(DEVICE)
)

attention_mask = (
    test_batch["attention_mask"]
    .to(DEVICE)
)

labels = (
    test_batch["labels"]
    .to(DEVICE)
)

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask,

            labels=labels
        )

print(
    "Initial MLM loss:",
    outputs.loss.item()
)

Initial MLM loss: 9.7913236618042


In [16]:
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)

In [17]:
NUM_EPOCHS = 1

GRAD_ACCUMULATION_STEPS = 8

MAX_GRAD_NORM = 1.0

NUM_TRAIN_BATCHES = len(
    train_loader
)

TOTAL_OPTIMIZER_STEPS = (
    NUM_TRAIN_BATCHES
    // GRAD_ACCUMULATION_STEPS
)

WARMUP_STEPS = int(
    0.1 *
    TOTAL_OPTIMIZER_STEPS
)

print(
    "Training batches:",
    NUM_TRAIN_BATCHES
)

print(
    "Optimizer steps:",
    TOTAL_OPTIMIZER_STEPS
)

print(
    "Warmup steps:",
    WARMUP_STEPS
)

Training batches: 155060
Optimizer steps: 19382
Warmup steps: 1938


In [18]:
def lr_lambda(current_step):

    if current_step < WARMUP_STEPS:

        return (
            current_step /
            max(1, WARMUP_STEPS)
        )

    return max(
        0.0,

        (
            TOTAL_OPTIMIZER_STEPS
            - current_step
        )
        /
        max(
            1,
            TOTAL_OPTIMIZER_STEPS
            - WARMUP_STEPS
        )
    )


scheduler = LambdaLR(
    optimizer,
    lr_lambda
)

In [19]:
scaler = torch.amp.GradScaler(
    "cuda"
)

In [20]:
model.train()

global_step = 0

running_loss = 0.0

optimizer.zero_grad(
    set_to_none=True
)

progress_bar = tqdm(

    train_loader,

    total=len(train_loader),

    desc="BERT Training"
)

for batch_idx, batch in enumerate(
    progress_bar
):

    input_ids = (
        batch["input_ids"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    attention_mask = (
        batch["attention_mask"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    labels = (
        batch["labels"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )

    # -----------------------------
    # Forward pass
    # -----------------------------

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask,

            labels=labels
        )

        loss = outputs.loss

        loss_for_backward = (
            loss /
            GRAD_ACCUMULATION_STEPS
        )

    # -----------------------------
    # Backward pass
    # -----------------------------

    scaler.scale(
        loss_for_backward
    ).backward()

    running_loss += (
        loss.item()
    )

    # -----------------------------
    # Optimizer step
    # -----------------------------

    if (
        (batch_idx + 1)
        % GRAD_ACCUMULATION_STEPS
        == 0
    ):

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        scheduler.step()

        optimizer.zero_grad(
            set_to_none=True
        )

        global_step += 1

        # -----------------------------
        # Logging
        # -----------------------------

        if global_step % 100 == 0:

            avg_loss = (
                running_loss /
                (
                    100 *
                    GRAD_ACCUMULATION_STEPS
                )
            )

            current_lr = (
                scheduler
                .get_last_lr()[0]
            )

            progress_bar.set_postfix(

                loss=f"{avg_loss:.4f}",

                lr=f"{current_lr:.2e}"
            )

            running_loss = 0.0

BERT Training:   0%|          | 0/155060 [00:00<?, ?it/s]

In [21]:
BERT_DIR = (
    "/kaggle/working/"
    "indian_legal_bert"
)

os.makedirs(
    BERT_DIR,
    exist_ok=True
)

model.save_pretrained(
    BERT_DIR
)

tokenizer.save_pretrained(
    BERT_DIR
)

torch.save(
    optimizer.state_dict(),
    os.path.join(
        BERT_DIR,
        "optimizer.pt"
    )
)

torch.save(
    scheduler.state_dict(),
    os.path.join(
        BERT_DIR,
        "scheduler.pt"
    )
)

print(
    "BERT saved to:",
    BERT_DIR
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT saved to: /kaggle/working/indian_legal_bert


In [22]:
model.eval()

total_validation_loss = 0.0

validation_steps = 0

progress_bar = tqdm(

    validation_loader,

    total=len(validation_loader),

    desc="BERT Validation"
)

with torch.no_grad():

    for batch in progress_bar:

        input_ids = (
            batch["input_ids"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        attention_mask = (
            batch["attention_mask"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        labels = (
            batch["labels"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask,

                labels=labels
            )

        total_validation_loss += (
            outputs.loss.item()
        )

        validation_steps += 1

validation_loss = (
    total_validation_loss /
    validation_steps
)

print(
    "Validation MLM Loss:",
    validation_loss
)

BERT Validation:   0%|          | 0/17474 [00:00<?, ?it/s]

Validation MLM Loss: 6.007112337274847


In [23]:
correct = 0
total = 0

model.eval()

progress_bar = tqdm(

    validation_loader,

    total=len(validation_loader),

    desc="BERT MLM Accuracy"
)

with torch.no_grad():

    for batch in progress_bar:

        input_ids = (
            batch["input_ids"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        attention_mask = (
            batch["attention_mask"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        labels = (
            batch["labels"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits = model(

                input_ids=input_ids,

                attention_mask=attention_mask
            ).logits

        predictions = logits.argmax(
            dim=-1
        )

        mask = (
            labels != -100
        )

        correct += (
            (
                predictions[mask]
                ==
                labels[mask]
            )
            .sum()
            .item()
        )

        total += (
            mask.sum()
            .item()
        )

mlm_accuracy = (
    correct /
    total
)

print(
    "MLM Accuracy:",
    mlm_accuracy
)

print(
    "MLM Accuracy (%):",
    mlm_accuracy * 100
)

BERT MLM Accuracy:   0%|          | 0/17474 [00:00<?, ?it/s]

MLM Accuracy: 0.15998776979360196
MLM Accuracy (%): 15.998776979360196


In [24]:
results = {

    "architecture": "BERT",

    "objective": "Masked Language Modeling",

    "vocab_size": VOCAB_SIZE,

    "context_length": CONTEXT_LENGTH,

    "hidden_size": D_MODEL,

    "num_attention_heads": NUM_HEADS,

    "num_hidden_layers": NUM_LAYERS,

    "intermediate_size": D_FF,

    "parameters": model.num_parameters(),

    "train_examples": len(
        dataset["train"]
    ),

    "validation_examples": len(
        dataset["validation"]
    ),

    "epochs": NUM_EPOCHS,

    "learning_rate": LEARNING_RATE,

    "weight_decay": WEIGHT_DECAY,

    "gradient_accumulation_steps":
        GRAD_ACCUMULATION_STEPS,

    "validation_mlm_loss":
        validation_loss,

    "validation_mlm_accuracy":
        mlm_accuracy
}

with open(
    os.path.join(
        BERT_DIR,
        "results.json"
    ),
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )

print(
    json.dumps(
        results,
        indent=4
    )
)

{
    "architecture": "BERT",
    "objective": "Masked Language Modeling",
    "vocab_size": 16001,
    "context_length": 2048,
    "hidden_size": 512,
    "num_attention_heads": 8,
    "num_hidden_layers": 6,
    "intermediate_size": 2048,
    "parameters": 28437121,
    "train_examples": 155060,
    "validation_examples": 17474,
    "epochs": 1,
    "learning_rate": 1e-05,
    "weight_decay": 0.01,
    "gradient_accumulation_steps": 8,
    "validation_mlm_loss": 6.007112337274847,
    "validation_mlm_accuracy": 0.15998776979360196
}
